# Joystick IMU: perceptrón multicapa para la velocidad (modo avanzado)
Mismas entradas que la regresión polinómica (angulo_abs, w_x, w_y) y mismo filtro de pico.
Se usa la base de datos completa (la que se amplió para el softmax) y la división se hace por ensayo.
Igual que en la dirección, se agregan muestras de postura sostenida (mano quieta en el pico).
La red se programa desde cero con NumPy (sin librerías de redes neuronales).

In [ ]:
# Cargar CSV desde Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

RUTA = '/content/drive/MyDrive/Sistemas inteligentes/Regresión Softmax Joystick/ensayos_imu.csv'   # base de datos completa
df = pd.read_csv(RUTA, encoding='latin1')
df['Direccion'] = df['Direccion'].astype(str).str.strip()
df.loc[df['Direccion'].str.lower().str.startswith('atr'), 'Direccion'] = 'atrás'   # arreglar tilde dañada
print('Muestras:', len(df))
print(df['Direccion'].value_counts())

In [ ]:
# Igual que la polinómica: solo adelante, atrás y N; ángulo de inclinación y su valor absoluto
df_roll = df[df['Direccion'].isin(['adelante', 'atrás', 'N'])].copy()
df_roll['angulo_roll'] = np.degrees(np.arctan2(df_roll['a_z'], -df_roll['a_x']))
df_roll['angulo_abs'] = df_roll['angulo_roll'].abs()

# Solo el pico de cada ensayo (85 % del ángulo máximo); un ensayo = usuario + trial
df_roll['grupo'] = df_roll['user_id'] * 100000 + df_roll['trial_id']
max_por_trial = df_roll.groupby('grupo')['angulo_abs'].transform('max')
df_pico = df_roll[df_roll['angulo_abs'] >= 0.85 * max_por_trial].copy()
print('Filas en el pico:', len(df_pico))

In [ ]:
# Entradas (angulo_abs, w_x, w_y) y salida (velocidad 0..1)
# División 80/20 por ensayo: las muestras de un mismo ensayo quedan juntas (sin fuga)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

X = df_pico[['angulo_abs', 'w_x', 'w_y']].values
y = df_pico['velocidad_objetivo'].values.reshape(-1, 1)
g = df_pico['grupo'].values
i_tr, i_te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X, y, g))

# Postura sostenida: cada muestra del pico se copia con el giro de la mano quieta
# (giroscopio real del inicio quieto del mismo ensayo); el ángulo y la velocidad no cambian
reposo = df_roll.groupby('grupo').head(10)
w_reposo = {k: v[['w_x', 'w_y']].values for k, v in reposo.groupby('grupo')}
rng = np.random.default_rng(0)
X_sost = np.column_stack([X[:, 0], [w_reposo[k][rng.integers(len(w_reposo[k]))] for k in g]])

X_train = np.vstack([X[i_tr], X_sost[i_tr]])
y_train = np.vstack([y[i_tr], y[i_tr]])
X_test, y_test = X[i_te], y[i_te]           # test con las muestras originales
X_test_s = X_sost[i_te]                     # test con posturas sostenidas

scaler = StandardScaler().fit(X_train)          # z-score con datos de train
Xtr, Xte, Xte_s = scaler.transform(X_train), scaler.transform(X_test), scaler.transform(X_test_s)
print('Train:', Xtr.shape, '| Test:', Xte.shape, '| Test sostenidas:', Xte_s.shape)

In [ ]:
# Red 3 -> 8 -> 8 -> 1
# Capas ocultas: tanh | Salida: sigmoide (deja la velocidad entre 0 y 1)
CAPAS = [3, 8, 8, 1]
rng = np.random.default_rng(0)
W = [rng.normal(0, np.sqrt(1 / CAPAS[i]), (CAPAS[i], CAPAS[i+1])) for i in range(3)]   # pesos
b = [np.zeros((1, CAPAS[i+1])) for i in range(3)]                                      # sesgos

sigmoide = lambda z: 1 / (1 + np.exp(-z))

def adelante(X):
    # Propagación hacia adelante: guarda la salida de cada capa
    a = [X]
    for i in range(3):
        z = a[-1] @ W[i] + b[i]
        a.append(sigmoide(z) if i == 2 else np.tanh(z))
    return a

In [ ]:
# Entrenamiento con gradiente descendente y retropropagación (costo: error cuadrático medio)
EPOCAS, LR = 4000, 0.5
costo_train, costo_test = [], []

for ep in range(EPOCAS):
    a = adelante(Xtr)
    m = len(Xtr)
    delta = (a[3] - y_train) * a[3] * (1 - a[3])      # error en la salida (derivada de la sigmoide)
    for i in (2, 1, 0):
        gW = a[i].T @ delta / m
        gb = delta.mean(0, keepdims=True)
        if i > 0:
            delta = (delta @ W[i].T) * (1 - a[i]**2)   # error que pasa a la capa anterior (derivada de tanh)
        W[i] -= LR * gW
        b[i] -= LR * gb
    costo_train.append(((adelante(Xtr)[3] - y_train)**2).mean() / 2)
    costo_test.append(((adelante(Xte)[3] - y_test)**2).mean() / 2)

print(f'Costo final -> train: {costo_train[-1]:.5f} | test: {costo_test[-1]:.5f}')

In [ ]:
# Curva de aprendizaje
plt.figure(figsize=(9, 4))
plt.plot(costo_train, label='Train'); plt.plot(costo_test, label='Test')
plt.xlabel('Época'); plt.ylabel('Costo (MSE/2)'); plt.grid(alpha=.4); plt.legend()
plt.title('Evolución del costo - MLP velocidad'); plt.show()

In [ ]:
# Métricas
from sklearn.metrics import mean_absolute_error, r2_score
p_tr, p_te = adelante(Xtr)[3], adelante(Xte)[3]
print(f'TRAIN -> MAE: {mean_absolute_error(y_train, p_tr):.4f} | R²: {r2_score(y_train, p_tr):.4f}')
print(f'TEST  -> MAE: {mean_absolute_error(y_test, p_te):.4f} | R²: {r2_score(y_test, p_te):.4f}')
p_te_s = adelante(Xte_s)[3]
print(f'TEST sostenidas -> MAE: {mean_absolute_error(y_test, p_te_s):.4f} | R²: {r2_score(y_test, p_te_s):.4f}')

plt.figure(figsize=(5, 5))
plt.scatter(y_test, p_te, s=10, color='black')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Velocidad real'); plt.ylabel('Velocidad predicha'); plt.grid(alpha=.4)
plt.title('Real vs. predicha (test)'); plt.show()

In [ ]:
# Comparación con la regresión polinómica del modo básico (θ del firmware), en el mismo test
MEAN_P  = np.array([38.1009651, 7.73562638, -1.76439703, 2122.29661, 448.215522, -143.665112, 437.159867, -274.597072, 1985.97891])
SCALE_P = np.array([25.89619806, 19.42472526, 44.5293814, 2189.85809529, 1286.45872902, 2274.87676336, 1313.35371098, 1229.23159976, 3294.56183187])
THETA_P = np.array([0.66657489, 0.80386927, -0.01309359, -0.14718238, -0.46418315, 0.01453122, 0.11901392, 0.00294823, 0.02212228, 0.0322368])
def poly(Xc):
    x0, x1, x2 = Xc.T
    P = np.column_stack([x0, x1, x2, x0**2, x0*x1, x0*x2, x1**2, x1*x2, x2**2])
    return np.clip(THETA_P[0] + ((P - MEAN_P) / SCALE_P) @ THETA_P[1:], 0, 1)

for nombre, Xc, p_mlp in [('Test', X_test, p_te), ('Test sostenidas', X_test_s, p_te_s)]:
    p_poly = poly(Xc)
    print(f'{nombre}:')
    print(f'  Polinómica -> MAE: {mean_absolute_error(y_test, p_poly):.4f} | R²: {r2_score(y_test, p_poly):.4f}')
    print(f'  MLP        -> MAE: {mean_absolute_error(y_test, p_mlp):.4f} | R²: {r2_score(y_test, p_mlp):.4f}')

# Velocidad según el ángulo con la mano quieta (w = 0)
ang = np.linspace(0, 60, 61)
Xq = np.column_stack([ang, np.zeros_like(ang), np.zeros_like(ang)])
plt.figure(figsize=(8, 4))
plt.plot(ang, adelante(scaler.transform(Xq))[3], 'k-', label='MLP')
plt.plot(ang, poly(Xq), 'k--', label='Polinómica')
plt.xlabel('Ángulo de inclinación (°)'); plt.ylabel('Velocidad'); plt.grid(alpha=.4); plt.legend()
plt.title('Velocidad con la mano quieta'); plt.show()

In [ ]:
# Pesos y sesgos (parámetros θ) y ecuación del modelo
np.set_printoptions(precision=5, suppress=True, linewidth=150)
for i in range(3):
    print(f'\nCapa {i+1}: W{i+1} {W[i].shape}\n{W[i]}\nb{i+1}: {b[i].ravel()}')
print('\nModelo:')
print('  x~ = (x - mu) / sigma            x = [angulo_abs, w_x, w_y]')
print('  h1 = tanh(x~ W1 + b1)            8 neuronas')
print('  h2 = tanh(h1 W2 + b2)            8 neuronas')
print('  velocidad = sigmoide(h2 W3 + b3)')

In [ ]:
# Bloque para pegar en el .ino
f = lambda v: ', '.join(f'{x:.7f}f' for x in np.ravel(v))
mat = lambda nombre, M: f'const float {nombre}[{M.shape[0]}][{M.shape[1]}] = {{\n' + \
      ',\n'.join('  { ' + f(fila) + ' }' for fila in M) + '\n};'
bloque = '\n'.join([
    '// Generado por el notebook MLP velocidad. No editar a mano.',
    f'const float MV_MEAN[3] = {{ {f(scaler.mean_)} }};',
    f'const float MV_SCALE[3] = {{ {f(scaler.scale_)} }};',
    mat('MV_W1', W[0]), f'const float MV_B1[8] = {{ {f(b[0])} }};',
    mat('MV_W2', W[1]), f'const float MV_B2[8] = {{ {f(b[1])} }};',
    mat('MV_W3', W[2]), f'const float MV_B3[1] = {{ {f(b[2])} }};',
])
print(bloque)
with open('parametros_mlp_velocidad.txt', 'w') as fh:
    fh.write(bloque + '\n')
from google.colab import files
files.download('parametros_mlp_velocidad.txt')